# Run ENCflow in your browser (no installation)

This notebook is an introduction to
**[ENCflow](https://github.com/ENCflow/ENCflow)** — a flood and
catchment-hydrology simulation program — running on Google Colab
without installing anything on your computer.

[日本語版はこちら](https://colab.research.google.com/github/ENCflow/ENCflow/blob/main/docs/colab_quickstart.ipynb)

- Just run the cells from top to bottom (press the ▶ at the left edge
  of a cell, or `Shift+Enter`).
- The whole notebook takes about 5 minutes.
- Lines starting with `!` or `%%bash` execute Linux commands as-is.
  The same commands will work later on your own machine (WSL etc.;
  see [Using ENCflow on Windows](https://github.com/ENCflow/ENCflow/blob/main/docs/en/windows.md)).


## 1. Setup: compile the executable

ENCflow is a Fortran program with zero external libraries, so it
builds anywhere a compiler (gfortran) exists. This takes 1-2 minutes.


In [ ]:
%%bash
# Get a Fortran compiler, fetch ENCflow, and build it
apt-get -qq install -y gfortran > /dev/null
git clone --depth 1 -q https://github.com/ENCflow/ENCflow.git
cd ENCflow/src && make install -j2 2>&1 | tail -2


## 2. Your first simulation: a mound of water collapses and spreads

Run the minimal example (test/wave): a "mound of water" standing on a
still surface collapses and spreads in concentric circles. After the
run, `Run.sh` automatically compares the result **bit for bit**
against the reference bundled in the repository — `PASS` means your
environment produced exactly the same answer as the developers'.


In [ ]:
%%bash
cd ENCflow/test/wave && ./Run.sh 2>&1 | tail -3


## 3. Look at the result

The outputs are plain text matrices, so just read them with numpy and
draw them with matplotlib. `E0000.txt` ... `E0008.txt` are water-level
snapshots at 1-second intervals.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

fig, axes = plt.subplots(3, 3, figsize=(10, 10))
for k, ax in enumerate(axes.flat):
    e = np.loadtxt(f'ENCflow/test/wave/result/E{k:04d}.txt')
    im = ax.imshow(e, cmap='viridis', vmin=0.9, vmax=1.3)
    ax.set_title(f't = {k} s'); ax.axis('off')
fig.colorbar(im, ax=axes, shrink=0.6, label='water level (m)')
plt.show()


## 4. Build your own run: rain over a bowl-shaped terrain

The input to ENCflow is **nothing but text files**. Here we write out
a bowl-shaped terrain and a setting for a 30-minute, 100 mm/h
rainstorm from Python, and compute how the depression fills up.

Each line of the parameter file is explained by its comment. Change a
number and rerun — that is already your own numerical experiment
(e.g. rain at 200 mm/h, a deeper bowl, a larger roughness rn0...).


In [ ]:
import numpy as np

# --- Terrain: a bowl (0 m at the center, rising toward the rim) ---
n = 51
c = (n - 1) / 2
y, x = np.mgrid[0:n, 0:n]
z = 3.0 * ((x - c)**2 + (y - c)**2) / c**2
np.savetxt('z.txt', z, fmt='%.4f')

# --- Parameter file (this is the entire input to ENCflow) ---
param = '''
! Minimal example: rain ponding in a bowl-shaped terrain
&list_sysparam
  dt = 0.1                  ! time step (s)
  tt = 1800                 ! end time (s) = 30 minutes
  dt_disp = 180             ! screen display interval (s)
  dt_file = 180             ! file output interval (s)
  fn_geoinfo = '-'          ! '-' = read from this same file
  fn_initial = '-'
  fn_precip = '-'           ! rainfall = this line + the group below
  dir_data = '.'            ! where the input data (z.txt) is
  dir_result = 'result_rain'
/

&list_initial
  f_htype = 0               ! initial depth: fixed value
  h0 = 0.0                  ! start dry
/

&list_geoinfo
  lx = 102.0                ! domain size (m)
  ly = 102.0
  nx = 51                   ! number of cells
  ny = 51
  f_ztype = 1               ! terrain: from a file
  fn_z = 'z.txt'
  f_rntype = 0              ! roughness: fixed value
  rn0 = 0.05
/

&list_precip
  prtype = 1                ! uniform rainfall time series
  prval(1:2,1) = 0, 100     ! (min, mm/h): 100 mm/h from t = 0
  prval(1:2,2) = 9999, 100
/
'''
with open('param_rain.txt', 'w') as f:
    f.write(param)
print('wrote z.txt and param_rain.txt')


In [ ]:
%%bash
# Run (a few tens of seconds). After the log lines showing that the
# groups we wrote are being read, a legend line is printed followed by
# the progress every 3 minutes. The S(m) column (3rd column) is the
# domain-mean water storage - watch it grow linearly with the rain
# from 0 to 0.05 m (= 100 mm/h x 30 min)
./ENCflow/bin/encflow param_rain.txt


## 5. Animation of the ponding


In [ ]:
import glob
import matplotlib.pyplot as plt
from matplotlib import animation
from IPython.display import HTML

frames = [np.loadtxt(f) for f in sorted(glob.glob('result_rain/H0*.txt'))]
fig, ax = plt.subplots(figsize=(5, 5))
im = ax.imshow(frames[0], cmap='Blues', vmin=0, vmax=0.65)
ax.axis('off'); fig.colorbar(im, shrink=0.8, label='depth (m)')
ttl = ax.set_title('t = 0 min')
def update(k):
    im.set_data(frames[k]); ttl.set_text(f't = {k*3} min'); return [im, ttl]
anim = animation.FuncAnimation(fig, update, frames=len(frames), interval=400)
plt.close(fig)
HTML(anim.to_jshtml())


## Next steps

- **[Tutorial](https://github.com/ENCflow/ENCflow/blob/main/docs/tutorial.md)** —
  proceeds step by step up to a real-terrain catchment run
  (currently in Japanese).
- **[Use-case gallery](https://github.com/ENCflow/ENCflow/blob/main/docs/en/users_guide/usecases.md)** —
  look up the settings from the phenomenon you want to compute
  (pond breach, pluvial flooding, debris flow, ...).
- **Run it on your own machine** — Windows users: see
  [Using ENCflow on Windows](https://github.com/ENCflow/ENCflow/blob/main/docs/en/windows.md).
  The commands used in this notebook work there as-is.

Note: files in a Colab session disappear when the session is
disconnected. Download anything you want to keep from the file pane
on the left.
